<a href="https://colab.research.google.com/github/D2718281828nis/LLM_agent-FireCrawl-Graph/blob/main/FireCrawl-CR-site-parsing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Агентный сбор и анализ клинических рекомендаций

Notebook выполняет единый воспроизводимый сценарий:

1. Firecrawl Agent обходит динамический реестр Минздрава и сохраняет `ID`, `Наименование`, `Дата размещения КР`, `МКБ-10` в CSV.
2. Пользователь выбирает ID; Firecrawl извлекает алгоритмы действий врача из соответствующей рекомендации.
3. Mistral формирует структурированное резюме **только по извлечённому тексту**.

> **Медицинское предупреждение:** результат предназначен для исследовательского прототипа, может быть неполным и не заменяет официальный документ или решение врача. Всегда сверяйте вывод с исходной рекомендацией.


## 1. Установка зависимостей

Ключи должны называться `FIRECRAWL_API_KEY` и `MISTRAL_API_KEY`.

- **Google Colab**: добавьте их в Colab Secrets (значок ключа слева) и включите Notebook access.
- **VS Code / локально**: скопируйте `.env.example` в `.env` и укажите значения — файл `.env` игнорируется git'ом и не попадёт в репозиторий.

Не вставляйте значения ключей в код, вывод Notebook или чат.


In [ ]:
%pip install -q --upgrade firecrawl-py mistralai pandas pydantic langchain-mistralai langchain-core langchain-classic networkx matplotlib python-dotenv lxml tqdm

In [ ]:
from __future__ import annotations

import io
import json
import os
import re
import time
from pathlib import Path
from typing import Any, List, Dict, Optional # Added Optional
import networkx as nx # Added
import matplotlib.pyplot as plt # Added

import pandas as pd
from tqdm.auto import tqdm
from firecrawl import Firecrawl
# Removed: from mistralai import Mistral
from langchain_mistralai import ChatMistralAI # Added
from langchain_core.tools import tool # Added
from langchain_core.messages import HumanMessage # Added
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder # Added
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent # Added

from pydantic import BaseModel, Field

try:
    from google.colab import userdata # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    from dotenv import load_dotenv
    load_dotenv()

REGISTRY_URL = "https://cr.minzdrav.gov.ru/clin-rec"
DETAIL_URL_TEMPLATE = "https://cr.minzdrav.gov.ru/view-cr/{clinical_id}"
CSV_PATH = Path("clinical_recommendations_minzdrav.csv")
REQUIRED_COLUMNS = ["ID", "Наименование", "Дата размещения КР", "МКБ-10"]

def require_secret(name: str) -> str:
    value = userdata.get(name) if IN_COLAB else os.getenv(name)
    if not value:
        hint = (
            "Добавьте его в Colab Secrets и включите Notebook access."
            if IN_COLAB
            else "Скопируйте .env.example в .env и укажите значение."
        )
        raise RuntimeError(f"Секрет {name} не найден. {hint}")
    return value

firecrawl = Firecrawl(api_key=require_secret("FIRECRAWL_API_KEY"))

# Renamed 'mistral' to 'llm' and changed type to ChatMistralAI as per user's pattern
MODEL_NAME = 'mistral-small-latest'
llm = ChatMistralAI(
     model=MODEL_NAME,
     api_key=require_secret("MISTRAL_API_KEY"),
     temperature=0.1,
     max_retries=2,
)
print(f"✅ Клиенты настроены ({'Colab' if IN_COLAB else 'локально'}); значения секретов не выведены.")

## 2. Контракты данных и совместимость SDK

Pydantic-схемы заставляют агента вернуть машинно-читаемые поля. Вспомогательные функции нормализуют ответы разных версий Firecrawl SDK (`dict` или объект).


In [ ]:
class DoctorActions(BaseModel):
    title: str = Field(description="Наименование рекомендации")
    source_url: str
    source_sections: list[str] = Field(
        description="Заголовки разделов, из которых извлечены действия"
    )
    actions_text: str = Field(
        description="Полный текст алгоритмов и действий врача без резюмирования"
    )


def as_dict(result: Any) -> dict[str, Any]:
    """Convert Firecrawl/Pydantic responses to a plain dictionary."""
    if isinstance(result, dict):
        return result
    if hasattr(result, "model_dump"): # pydantic v2
        return result.model_dump()
    if hasattr(result, "dict"): # pydantic v1
        return result.dict()
    data = getattr(result, "data", None)
    if data is not None:
        return as_dict(data)
    raise TypeError(f"Неожиданный тип ответа: {type(result).__name__}")


def payload(result: Any) -> dict[str, Any]:
    """Unwrap the common Firecrawl {'data': ...} response envelope."""
    value = as_dict(result)
    return as_dict(value["data"]) if "data" in value else value

## 3. Обход реестра и CSV

У реестра **нет** кнопок «следующая страница» — обследование живого сайта показало,
что футер таблицы содержит только переключатель «Строк на странице» (5/10/25/50/100/
**Все**) без пагинации: если оставить маленький размер страницы, увидеть записи после
первой всё равно невозможно. Поэтому Firecrawl одним запросом кликает по этому
переключателю и выбирает «Все» — сайт отдаёт все 746 записей одной HTML-таблицей,
которую мы разбираем локально через `pandas.read_html` (без LLM — быстрее и без риска
галлюцинаций при чтении таблицы).

Дальше строки обрабатываются **последовательно, партиями**, с прогресс-баром и
сообщением о том, сколько и какие ID добавлены на каждом шаге — это и даёт видимый
пошаговый прогресс, который просили, при этом без лишних повторных запросов к сайту.
Повторный запуск обновляет CSV.


In [ ]:
REGISTRY_COUNTER_RE = re.compile(r"(\d+)\s*-\s*(\d+)\s*из\s*(\d+)")


def _fetch_registry_attempt(client: Firecrawl, settle_ms: int) -> tuple[str, Optional[tuple[int, int]]]:
    """Один запрос: открывает реестр, переключает 'Строк на странице' на 'Все'."""
    doc = client.scrape(
        REGISTRY_URL,
        formats=["html"],
        wait_for=2000,
        actions=[
            {"type": "click", "selector": ".border-t .v-field"},
            {"type": "wait", "milliseconds": 1000},
            {"type": "click", "selector": "[role='listbox'] > div:nth-last-child(2)"},
            {"type": "wait", "milliseconds": settle_ms},
        ],
    )
    html = as_dict(doc).get("html") or ""
    match = REGISTRY_COUNTER_RE.search(html)
    shown = (int(match.group(2)), int(match.group(3))) if match else None
    return html, shown


def _fetch_full_registry_html(client: Firecrawl = firecrawl, max_attempts: int = 4) -> str:
    """Гарантирует, что переключатель 'Строк на странице' реально встал на 'Все'.

    Клик по конкретному пункту меню — не 100% надёжная операция на живом сайте
    (сеть/анимация/лёгкий rate-limit могут привести к тому, что счётчик не
    обновится). Поэтому вместо одного клика "наудачу" сверяем счётчик
    "X - Y из Z" на странице и повторяем попытку, пока Y не сравняется с Z —
    иначе можно молча получить только первые 5 записей, как уже случалось.
    """
    last_shown: Optional[tuple[int, int]] = None
    last_error: Optional[Exception] = None
    for attempt in range(1, max_attempts + 1):
        settle_ms = 3000 + 1500 * (attempt - 1)
        try:
            html, shown = _fetch_registry_attempt(client, settle_ms)
        except Exception as e:
            last_error = e
            print(f"⚠️ Попытка {attempt}/{max_attempts} завершилась ошибкой запроса: {e}")
        else:
            last_error = None
            last_shown = shown
            if shown and shown[0] >= shown[1] > 0:
                if attempt > 1:
                    print(f"✅ Реестр полностью загружен с попытки {attempt}/{max_attempts} "
                          f"(показано {shown[0]} из {shown[1]}).")
                return html
            print(f"⚠️ Попытка {attempt}/{max_attempts}: переключатель 'Строк на странице' "
                  f"не применился (счётчик показывает {shown}), повторяем...")
        if attempt < max_attempts:
            time.sleep(4 * attempt)

    raise RuntimeError(
        "Не удалось переключить реестр на показ всех записей за "
        f"{max_attempts} попыток (последний показанный диапазон: {last_shown}"
        + (f", последняя ошибка: {last_error}" if last_error else "")
        + "). Данные не сохранены, чтобы не получить неполный CSV — запустите ячейку ещё раз."
    )


def _parse_registry_table(html: str) -> pd.DataFrame:
    tables = pd.read_html(io.StringIO(html))
    if not tables:
        raise RuntimeError("В HTML реестра не найдено ни одной таблицы.")
    raw = tables[0]
    missing = [c for c in REQUIRED_COLUMNS if c not in raw.columns]
    if missing:
        raise RuntimeError(f"В таблице реестра отсутствуют столбцы: {missing}")
    frame = raw.loc[:, REQUIRED_COLUMNS].copy()
    frame = frame.dropna(subset=["ID", "Наименование"])
    for col in REQUIRED_COLUMNS:
        frame[col] = frame[col].astype(str).str.strip()
    frame = frame[(frame["ID"] != "") & (frame["Наименование"] != "") & (frame["ID"] != "nan")]
    return frame.reset_index(drop=True)


def scrape_registry(client: Firecrawl = firecrawl, batch_size: int = 50) -> pd.DataFrame:
    print("⚙️ Получаем реестр клинических рекомендаций (переключаем 'Строк на странице' на 'Все')...")
    html = _fetch_full_registry_html(client)
    expected_total = REGISTRY_COUNTER_RE.search(html)
    expected_total = int(expected_total.group(3)) if expected_total else None

    frame = _parse_registry_table(html)
    total = len(frame)
    if total == 0:
        raise RuntimeError("Не удалось извлечь ни одной строки реестра.")
    if expected_total is not None and total != expected_total:
        raise RuntimeError(
            f"Счётчик сайта сообщил {expected_total} записей, а разбор таблицы дал {total} — "
            "часть строк могла потеряться при парсинге. Данные не сохранены, запустите ещё раз."
        )
    print(f"✅ Получено {total} строк реестра (совпадает со счётчиком сайта). "
          f"Обрабатываем последовательно, партиями по {batch_size}...")

    batches = []
    n_batches = -(-total // batch_size)  # ceil division
    for batch_num, start in enumerate(tqdm(range(0, total, batch_size), total=n_batches, desc="Обход реестра", unit="партия"), start=1):
        batch = frame.iloc[start:start + batch_size]
        batches.append(batch)
        ids = ", ".join(batch["ID"])
        print(f"✅ Партия {batch_num}/{n_batches}: добавлено {len(batch)} записей (ID: {ids}).")

    processed = pd.concat(batches, ignore_index=True)
    processed = processed.drop_duplicates(subset="ID", keep="last").reset_index(drop=True)
    if processed.empty:
        raise RuntimeError("После проверки данных реестр оказался пустым.")
    if len(processed) != total:
        print(f"⚠️ Внимание: после удаления дублей осталось {len(processed)} из {total} строк "
              f"(были повторяющиеся ID).")
    return processed


registry_df = scrape_registry()
print(f"💾 Сохраняем данные в {CSV_PATH.name}...")
registry_df.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
print(f"✅ Сохранено {len(registry_df)} уникальных, "
      f"валидных строк в: {CSV_PATH.resolve()}")
display(registry_df.head(10))

In [ ]:
# В Colab дополнительно скачиваем файл; локально он уже сохранён на диск по CSV_PATH.
if IN_COLAB:
    from google.colab import files
    files.download(str(CSV_PATH))
else:
    print(f"📄 Файл сохранён локально: {CSV_PATH.resolve()}")

## 4. Выбор ID и извлечение действий врача

ID валидируется по уже собранному реестру, поэтому произвольный URL создать нельзя. Агент ищет прежде всего «Приложение Б. Алгоритмы действий врача», а также связанные разделы диагностики, лечения, диспансерного наблюдения и критериев срочного направления.


In [ ]:
def choose_clinical_id(frame: pd.DataFrame) -> str:
    known_ids = set(frame["ID"].astype(str))
    while True:
        clinical_id = input("Введите ID клинической рекомендации: ").strip()
        if clinical_id in known_ids:
            row = frame.loc[frame["ID"].astype(str) == clinical_id].iloc[0]
            print(f"Выбрано: {row['Наименование']}")
            return clinical_id
        print("ID отсутствует в загруженном CSV. Проверьте значение и повторите ввод.")


clinical_id = choose_clinical_id(registry_df)

In [ ]:
def extract_doctor_actions(
    clinical_id: str, client: Firecrawl = firecrawl
) -> DoctorActions:
    if not re.fullmatch(r"[A-Za-zА-Яа-яЁё0-9_-]+", clinical_id):
        raise ValueError("ID содержит недопустимые символы.")
    url = DETAIL_URL_TEMPLATE.format(clinical_id=clinical_id)
    prompt = f"""
Проанализируй только официальный документ по адресу {url}. Найди «Приложение Б.
Алгоритмы действий врача» и извлеки его полный текст без пересказа. Если приложение
отсутствует или недоступно, извлеки точные фрагменты с действиями врача из разделов
диагностики, лечения, медицинской помощи, диспансерного наблюдения и критериев
срочного направления. Сохрани порядок, условия, отрицания, дозировки и уровни
рекомендаций. Перечисли реально использованные заголовки разделов. Не добавляй
медицинские сведения, которых нет на странице. source_url должен быть равен {url}.
""".strip()
    result = client.agent(
        prompt=prompt, urls=[url], schema=DoctorActions.model_json_schema()
    )
    actions = DoctorActions.model_validate(payload(result))
    if not actions.actions_text.strip():
        raise RuntimeError("Не удалось извлечь действия врача из документа.")
    return actions


actions = extract_doctor_actions(clinical_id)
print(f"✅ Извлечено символов: {len(actions.actions_text)}")
print(f"Источник: {actions.source_url}")
print("Разделы:", "; ".join(actions.source_sections))

## 5. Agentic summary с Mistral

Mistral получает исходный текст и строгие правила: не дополнять его внешними знаниями, отделять условия и указывать пробелы. Ответ сохраняет ссылку на официальный документ для проверки.


In [ ]:
SUMMARY_SYSTEM_PROMPT = """
Ты — аналитический агент по клиническим рекомендациям. Работай ИСКЛЮЧИТЕЛЬНО с
переданным текстом. Не ставь диагноз, не назначай лечение и не дополняй ответ
внешними медицинскими знаниями. Если данных нет, явно пиши «не указано в
извлечённом фрагменте». Сохраняй отрицания, условия, дозировки и срочность.
Ответ дай на русском языке в Markdown со структурой:
1. Цель алгоритма
2. Последовательность действий врача (нумерованный список)
3. Условия и точки принятия решений (если → то)
4. Красные флаги и срочные действия
5. Контроль, наблюдение и критерии эскалации
6. Что требует проверки в полном официальном документе
В конце добавь предупреждение, что это исследовательское резюме, а не медицинская
рекомендация.
""".strip()


def summarize_actions(actions: DoctorActions, client: ChatMistralAI = llm) -> str:
    source = json.dumps(actions.model_dump(), ensure_ascii=False, indent=2)
    response = client.invoke(
        [
            ("system", SUMMARY_SYSTEM_PROMPT),
            ("human", f"Сформируй резюме этого извлечения:\n{source}"),
        ]
    )
    summary = response.content
    if not summary:
        raise RuntimeError("Mistral вернул пустое резюме.")
    return summary


summary = summarize_actions(actions)
print(summary)
print(f"\nОфициальный источник: {actions.source_url}")

## 6. Сохранение аудита

Помимо CSV сохраняются необработанное извлечение и резюме. Это позволяет эксперту сопоставить вывод Mistral с текстом, который получил агент.


In [ ]:
audit_path = Path(f"clinical_recommendation_{clinical_id}_analysis.json")
audit_path.write_text(
    json.dumps(
        {
            "clinical_id": clinical_id,
            "extraction": actions.model_dump(),
            "summary": summary,
            "disclaimer": "Исследовательский прототип; требуется проверка врачом и по официальному документу.",
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)
print(f"✅ Аудит сохранён: {audit_path.resolve()}")

if IN_COLAB:
    from google.colab import files
    files.download(str(audit_path))